## 01 — Using Tippecanoe

`tippecanoe` is a command-line tool from Mapbox that converts GeoJSON into a vector tile pyramid.

One command replaces our entire Module 02 pipeline — and produces a smaller, faster output format. This notebook runs it, inspects the output, and maps every flag to a decision we already made by hand.

## Installation

On macOS with Homebrew:

```bash
brew install tippecanoe
```

On Linux (Ubuntu/Debian):

```bash
sudo apt-get install tippecanoe
```

Verify the install:

In [2]:
import os
import sys
import shutil
from pathlib import Path

SCRIPTS_DIR = str(Path(sys.executable).parent)
os.environ["PATH"] = SCRIPTS_DIR + os.pathsep + r"C:\Windows\System32" + os.pathsep + os.environ["PATH"]

print("Scripts dir:", SCRIPTS_DIR)
print("tippecanoe:", shutil.which("tippecanoe"))
print("tile-join:", shutil.which("tile-join"))
print("wsl.exe:", shutil.which("wsl.exe"))

Scripts dir: c:\Users\shara\Downloads\data spati\clean_env\Scripts
tippecanoe: c:\Users\shara\Downloads\data spati\clean_env\Scripts\tippecanoe.BAT
tile-join: c:\Users\shara\Downloads\data spati\clean_env\Scripts\tile-join.BAT
wsl.exe: C:\Windows\System32\wsl.exe


In [6]:
import os
import sys
import shutil
from pathlib import Path

scripts = str(Path(sys.executable).parent / "Scripts")
os.environ["PATH"] = scripts + os.pathsep + os.environ["PATH"]

print("Scripts folder:", scripts)
print("tippecanoe found at:", shutil.which("tippecanoe"))
print("tile-join found at:", shutil.which("tile-join"))

Scripts folder: c:\Users\shara\Downloads\data spati\clean_env\Scripts\Scripts
tippecanoe found at: c:\Users\shara\Downloads\data spati\clean_env\Scripts\tippecanoe.BAT
tile-join found at: c:\Users\shara\Downloads\data spati\clean_env\Scripts\tile-join.BAT


In [ ]:
import subprocess

result = subprocess.run(
    "tippecanoe --version",
    capture_output=True,
    text=True,
    shell=True
)

print(result.stdout or result.stderr)

In [ ]:
import subprocess
result = subprocess.run(["tippecanoe", "--version"], capture_output=True, text=True)
print(result.stdout or result.stderr)

tippecanoe v1.36.0



## Running Tippecanoe

The basic command:

```bash
tippecanoe \
  --output=railroads.pmtiles \
  --minimum-zoom=1 \
  --maximum-zoom=14 \
  --simplification=10 \
  --drop-densest-as-needed \
  --layer=railroads \
  ne_10m_railroads.geojson
```

Let's run it from Python and capture the output:

In [8]:
from pathlib import Path
import subprocess
import time

input_file  = Path("../../data/ne_10m_railroads.geojson")
output_file = Path("../../data/railroads.pmtiles")

cmd = [
    "tippecanoe",
    f"--output={output_file}",
    "--force",                     # overwrite if exists
    "--minimum-zoom=1",
    "--maximum-zoom=14",
    "--simplification=10",         # Douglas-Peucker tolerance in tile pixels
    "--drop-densest-as-needed",    # drop features at low zoom if tile is too large
    "--layer=railroads",
    str(input_file),
]

t0 = time.perf_counter()
result = subprocess.run(cmd, capture_output=True, text=True)
elapsed = time.perf_counter() - t0

print(result.stderr)   # tippecanoe writes progress to stderr
print(f"\nCompleted in {elapsed:.1f}s")

FileNotFoundError: [WinError 2] The system cannot find the file specified

## Inspecting the Output

In [ ]:
size_mb = output_file.stat().st_size / 1_000_000
raw_mb  = input_file.stat().st_size  / 1_000_000

print(f"Input  (raw GeoJSON):  {raw_mb:.1f} MB")
print(f"Output (PMTiles):      {size_mb:.2f} MB")
print(f"Compression ratio:     {raw_mb / size_mb:.1f}×  smaller")

## Mapping Flags to Decisions We Already Made

Every `tippecanoe` flag corresponds to something we built or decided manually:

| tippecanoe flag | What it does | Our equivalent |
|-----------------|-------------|----------------|
| `--minimum-zoom` | First zoom level that gets tiles | Bottom of our LOD range |
| `--maximum-zoom` | Most detailed zoom level | Top of our LOD range |
| `--simplification=10` | D-P tolerance in tile pixels per zoom | Our epsilon per LOD level |
| `--drop-densest-as-needed` | Remove least-important features when tile is too large | Our `scalerank <= 4` coarse filter |
| `--layer=railroads` | Names the data layer in the tile | Our filename convention |

The flags we do NOT have to specify:
- Viewport culling — built into the tile addressing scheme
- Binary encoding — automatic (MVT format)
- Tile pyramid structure — automatic
- Spatial index — automatic (tiles ARE the index)
- Zoom-driven switching — automatic (client requests the right `{z}` tiles)


## Inspecting Tile Contents with sqlite3

PMTiles can be converted to `.mbtiles` (SQLite) for inspection. Or we can use the `pmtiles` CLI to peek at specific tiles.

Alternatively, inspect the metadata embedded in the PMTiles file:

In [ ]:
# Use tippecanoe's companion tool to show metadata
result = subprocess.run(
    ["tile-join", "--no-tile-compression", "--if-matched",
     f"--output={output_file.with_suffix('.inspect.pmtiles')}",
     str(output_file)],
    capture_output=True, text=True
)

# Simpler: just show what pmtiles show gives us
result2 = subprocess.run(
    ["pmtiles", "show", str(output_file)],
    capture_output=True, text=True
)
print(result2.stdout or result2.stderr or "(install pmtiles CLI: pip install pmtiles)")

## Viewing in ipyleaflet

ipyleaflet supports PMTiles through the `PMTilesLayer` (requires `ipyleaflet >= 0.18`).

For local files, we need to serve them via a local HTTP server or use `localtileserver`.

In [ ]:
# Try loading with localtileserver if available
try:
    from localtileserver import TileClient, get_leaflet_tile_layer
    from ipyleaflet import Map

    client = TileClient(str(output_file))
    layer  = get_leaflet_tile_layer(client)
    m = Map(center=client.center(), zoom=client.default_zoom)
    m.add(layer)
    m
except ImportError:
    print("localtileserver not installed.")
    print("Install with: pip install localtileserver")
    print()
    print("Alternative: upload railroads.pmtiles to https://pmtiles.io to view it online.")
    print(f"File location: {output_file.resolve()}")

## Exercise A

Run `tippecanoe` a second time with `--maximum-zoom=8` and compare the output file size.

Then answer: what did limiting the maximum zoom cost us in terms of user experience, and what did it save?

In [ ]:
# Run tippecanoe with --maximum-zoom=8 and compare output size

output_file_z8 = Path("../../data/railroads_z8.pmtiles")

cmd2 = [
    "tippecanoe",
    f"--output={output_file_z8}",
    "--force",
    "--minimum-zoom=1",
    "--maximum-zoom=8",
    "--simplification=10",
    "--drop-densest-as-needed",
    "--layer=railroads",
    str(input_file),
]

t0 = time.perf_counter()
result2 = subprocess.run(cmd2, capture_output=True, text=True)
elapsed2 = time.perf_counter() - t0

print(result2.stderr)
print(f"\nCompleted in {elapsed2:.1f}s")

size_mb_z14 = output_file.stat().st_size / 1_000_000
size_mb_z8  = output_file_z8.stat().st_size / 1_000_000

print()
print(f"Output with max zoom 14: {size_mb_z14:.2f} MB")
print(f"Output with max zoom 8:  {size_mb_z8:.2f} MB")
print(f"Saved:                   {size_mb_z14 - size_mb_z8:.2f} MB")

# Lowering the maximum zoom should make the file smaller,
# because tippecanoe does not have to store the most detailed tiles.
# What it saves is storage space and less data to transfer.
# What it costs is close-up detail, because the map cannot serve very detailed
# tiles above zoom 8. So the map should be lighter, but it will look more
# generalized when the user zooms in.

## Exercise B

The `--simplification=10` flag sets the tolerance in **tile pixels**, not degrees. At zoom 14, a tile covers roughly 2.4km × 2.4km in 4096 pixels — so one pixel ≈ 0.6m.

Calculate what `--simplification=10` means in meters at zoom levels 2, 5, 8, and 12. Compare these to the degree-based epsilon values we chose in Module 02.

In [ ]:
# Calculate simplification tolerance in meters at different zoom levels
# Compare to our Module 02 epsilon choices

# From the notes:
# at zoom 14, one pixel is about 0.6 m
# so 10 pixels is about 6 m

meters_per_pixel_z14 = 0.6
zoom_levels = [2, 5, 8, 12]

print(f"{'Zoom':>4} {'Meters/pixel':>14} {'10-pixel simplification':>24}")
print("-" * 46)

for z in zoom_levels:
    meters_per_pixel = meters_per_pixel_z14 * (2 ** (14 - z))
    simplification_m = meters_per_pixel * 10
    print(f"{z:>4} {meters_per_pixel:>14.1f} {simplification_m:>24.1f}")

# Compared to Module 02, our epsilon values were fixed in degrees.
# tippecanoe is different because its simplification changes with zoom.
# That makes it more scale-aware: very large tolerance at low zoom,
# and much smaller tolerance at high zoom.

## Check Your Understanding

We ran `tippecanoe` with `--drop-densest-as-needed`. This flag tells tippecanoe to automatically drop the least-important features when a tile would otherwise be too large.

How does tippecanoe decide which features are "least important"? And how does that compare to our manual `scalerank <= 4` filter? Which approach is more principled — and what are the tradeoffs of each?

---

`tippecanoe` does not use one fixed rule like our `scalerank <= 4` filter. With `--drop-densest-as-needed`, it drops features automatically when a tile gets too crowded, so the decision depends on tile density and zoom level. That makes the tippecanoe approach more principled for tile generation, because it adapts to local conditions instead of using the same cutoff everywhere. Our manual filter was simpler and easier to control, but it was also more blunt because it applied one fixed importance rule across the whole dataset.

## Next

In [02 — The Comparison](./02-The_Comparison.ipynb), we put both systems side by side and answer the final question: what did `tippecanoe` actually save us from?